# Атака через другую кривую

## Операция сложения на группе точек эллиптической кривой

Чтобы понять принцип работы этой атаки, придется немного погрузиться в математику сложения точек на кривой. По какой формуле складываются точки на кривых Вейерштрассе? Дано короткое уравнение Вейерштрассе
$$y^2=x^3+Ax+B$$
и две точки $P$ и $Q$, которые нужно сложить. Давайте сначала рассмотрим особые случаи.

1. Если $P=0$, то $P+Q=Q$. Если $Q=0$, то $P+Q=P$

2. Если $P=-Q$, то $P+Q=0$.

Осталось только два общих случая:

1. $P \ne Q$

2. $P=Q$

Давайте разберёмся с первым.

![Addition](additionexample.png)

Что нам нужно сделать? Провести прямую сквозь $P$ и $Q$, найти третье пересечение $R$ с кривой и вычислить обратную ему точку. Значит, нужно найти уравнение прямой $y=mx+c$.

$$m=\frac{Q_y-P_y}{Q_x-P_x}$$

$$c=Q_y-mQ_x$$

Раз мы знаем уравнение прямой, можно заменить $y$ в уравнении эллиптической кривой на $mx+c$.

$$(mx+c)^2=x^3+Ax+B$$

$$x^3-(mx+c)^2+Ax+B=0$$

$$x^3-m^2x^2-2mcx-c^2+Ax+B=0$$

$$x^3- m^2x^2+(A-2mc)x+(B-c^2)=0$$

Мы также знаем, что:

$$x^3- m^2x^2+(A-2mc)x+(B-c^2)=(x-P_x)(x-Q_x)(x-R_x)$$

$$(x-P_x)(x-Q_x)(x-R_x)=(x^2-(P_x+Q_x)x+P_xQ_x)(x-R_x)=...$$
$$...=x^3-(P_x+Q_x)x^2+P_xQ_xx-R_xx^2+(P_x+Q_x)R_xx-P_xQ_xR_x=...$$
$$...=x^3-(P_x+Q_x+R_x)x^2+(P_xQ_x+P_xR_x+Q_xR_x)x-P_xQ_xR_x$$

$$x^3- m^2x^2+(A-2mc)x+(B-c^2)=x^3-(P_x+Q_x+R_x)x^2+(P_xQ_x+P_xR_x+Q_xR_x)x-P_xQ_xR_x$$

Значит

$$m^2=P_x+Q_x+R_x$$

$$R_x=m^2-P_x-Q_x$$

$$R_y=m*R_x+c$$

$$P+Q=(R_x,-R_y)$$

Теперь переключимся на следующий случай, когда $P=Q$. Нужно найти касательную к эллиптической кривой в точке $P$.

![Point Doubling](pointdoubling.png)

$$\frac{y^2}{\partial x}=\frac{x^3+Ax+B}{\partial x}$$

$$\frac{y^2\cdot\partial y}{\partial y\cdot\partial x}=3x^2+A$$

$$\frac{y^2}{\partial y}\frac{\partial y}{\partial x}=3x^2+A$$

$$2y\frac{\partial y}{\partial x}=3x^2+A$$

$$\frac{\partial y}{\partial x}=\frac{3x^2+A}{2y}$$

И $$m=\frac{\partial y}{\partial x}=\frac{3x^2+A}{2y}$$

Значит

$$y=mx+c$$

$$c=P_y-mP_x$$

$$R_x=m^2-2*P_x$$

$$R_y=m*R_x+c$$

$$2P=(R_x,-R_y)$$

Очевидно, все операции выполняются в поле, на котором определена эллиптическая кривая. В нашем случае на  $F_p$.

## Уязвимость
Возможно Вы заметили одну особенность. Когда мы складываем две точки, мы никогда не используем в вычислениях $B$, только $A$ для удваивания. Информация о $B$ содержится в самих точках.

Представьте, что сервер использует безопасную эллиптическую кривую для ECDH (и не эфемерный вариант, так что открытый и закрытый ключ не меняются). Вы отсылаете свой открытый ключ и оба вычисляете ключ для симметричного шифрования. Порядок группы точек на кривой $y^2=x^3+Ax+B$ - это  простое число $q$, так что подгрупп, которые можно было бы использовать, нет. Но если нет одной ключевой проверки перед скалярным умножением точки, защиту можно обойти. Вышеуказанная проверка - это проверка на принадлежность точки кривой, т.е. удовлетворяют ли её координаты заданному уравнению кривой.

Если эта проверка не проводится, злоумышленник может отправить точку с другой кривой, с намного более удобным для факторизации порядком группы (необходимо только, чтобы у кривых совпадал коэффициент $A$). Это Вам и предстоит использовать.

## Задание
Уравнение эллиптической кривой, которую использует сервер - это $y^2=x^3+4x+221$ определенное в $GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)$.

Порядок группы - $113311125625150355228527808652129931021724931561015845843493194156576487486212$, и он раскладывается на $$2^2 \cdot 28327781406287588807131952163032482755431232890253961460873298539144121871553$$ Так что подгруппы явно не особо помогут.

Однако сервер позволяет отправлять ему точки $P$ и возвращает $a\cdot P$ без каких-либо проверок, что $P$ принадлежит выбранной эллиптической кривой.

У порядков групп точек на следующих кривых достаточно малых (<65536) делителей, чтобы их $КТО$ было больше, чем порядок группы изначальной кривой.

+ $y^2 = x^3 + 4*x + 142$ с порядком группы\
$113311125625150355228527808652129931021494535848207968629801995728793690675887$

+ $y^2 = x^3 + 4*x + 254$ с порядком группы\
$113311125625150355228527808652129931021927787708044781007210395786720384091850$

+ $y^2 = x^3 + 4*x + 257$ с порядком группы\
$113311125625150355228527808652129931021320700166477893912982394278528617798310$

+ $y^2 = x^3 + 4*x + 277$ с порядком группы\
$113311125625150355228527808652129931022405298874054957089156266046101116988928$

+ $y^2 = x^3 + 4*x + 286$ с порядком группы\
$113311125625150355228527808652129931021623590516711700884680922146557735250848$

Вам нужно найти точки на кривых, ввести их в соответствующие подгруппы и получить остатки от закрытого ключа, используемого сервером.

Чтобы найти точки, выберите случайное $x$ в $GF(p)$, вычислите $y^2$, проверьте, что $(y^2)^{\frac{p-1}{2}} = 1\ mod\ p$, и если это так, $y_1=(y^2)^{\frac{p+1}{4}}$, $y_2=p-y_1$.

Я крайне рекомендую Вам сначала попробовать решить для собственного известного закрытого ключа $k$, и лишь когда Вы автоматизируете весь процесс, найти закрытый ключ, генерируемый сервером.

Удачи!

In [1]:
class IllegalScalar(Exception):
    pass
class EllipticCurve:
    def __init__(self, p, a, b):
        self.p = p
        self.a = a % p
        self.b = b % p

    def __eq__(self, oc):
        return self.p == oc.p and self.a == oc.a and self.b == oc.b

    def __str__(self):
        a,b=self.a,self.b
        a_str= '' if a==0 else str(a)+'*x' if a<0 else '+'+str(a)+'*x'
        b_str= '' if b==0 else str(b) if a<0 else '+'+str(b)

        return f"E(GF({self.p})) for y**2=x**3"+a_str+b_str

    def __repr__(self):
        return self.__str__()
class Point:
    def __init__(self, curve, x, y,is_identity=False):
        self.x = x % curve.p
        self.y = y % curve.p
        self.curve = curve
        self._is_identity=is_identity

    def is_identity(self):
        return self._is_identity

    def __eq__(self, Q):
        return (self._is_identity and Q._is_identity) or ( self.x == Q.x and self.y == Q.y and self.curve == Q.curve)

    def __str__(self):
        if self._is_identity:
            return "Точка на бесконечности на "+str(self.curve)
        return "({0},{1}) on ".format(self.x, self.y)+str(self.curve)

    def __repr__(self):
        return self.__str__()

    def __neg__(self):
        return Point(self.curve, self.x, self.curve.p-self.y,self._is_identity)

    def __add__(self, Q):
        if self.is_identity():

            return Q
        if Q.is_identity():

            return self
        if self.x == Q.x and self.y == self.curve.p-Q.y:
            return Point(self.curve, 0, 1,True)
        if self == Q:
            m = (((3*pow(self.x, 2, self.curve.p)+self.curve.a) %
                  self.curve.p)*pow(2*self.y, self.curve.p-2, self.curve.p)) % self.curve.p
        else:
            m = ((Q.y-self.y)*pow(Q.x -
                                  self.x, self.curve.p-2, self.curve.p)) % self.curve.p
        x3 = (pow(m, 2, self.curve.p)-self.x-Q.x) % self.curve.p
        y3 = (m*(self.x-x3)-self.y) % self.curve.p
        return Point(self.curve, x3, y3)

    def __mul__(self, k):
        if type(k) != int:
            raise IllegalScalar
        negate=False
        if k < 0:
            k=-k
            negate=True
        if k == 0:
            return Point(self.curve, 0, 1,True)
        bp = Point(self.curve, 0, 1,True)
        if negate:
            P = -self
        else:
            P=self
        i = 0
        while k != 0:
            if k & 1 != 0:
                bp = bp+P
            k >>= 1
            P = P+P
        return bp

    def __rmul__(self, k):
        return self.__mul__(k)


In [2]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключение к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1348))

    def recv_until(self,symb=b'\n>'):
        """Получение сообщений от сервера, по умолчанию до приглашения"""
        data=b''
        while True:

            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def getChallenge(self,show=True):
        """Получить параметры задания с сервера"""
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования Юникода. Попытайтесь переподключиться к серверу.')
            return (None,None)
        if show:
            print (data)
        (Gx,Gy)=tuple(map(int,re.search(r'(?<=G=\()\d+,\d+(?=\))',data).group(0).split(',')))
        p=int(re.search(r'(?<=E\(GF\()\d+(?=\)\))',data).group(0))
        group_order=int(re.search(r'(?<=order )\d+',data).group(0))
        axb=re.search(r'(?<=x\*\*3)([+-]\d+\*x)?([+-]\d+)?',data).group(0)
        ax=re.search(r'([+-]\d+\*x)?',axb)

        a=0 if ax==None else int(ax.group(0)[:-2])
        b=re.search(r'([+-]\d+)?$',axb)
        b=0 if b==None else int(b.group(0))
        (Qx,Qy)=tuple(map(int,re.search(r'(?<=Q=\()\d+,\d+(?=\))',data).group(0).split(',')))
        return (Gx,Gy,p,a,b,group_order,Qx,Qy)

    def checkSolution(self,k, show=True):
        """Проверка k (показывает флаг, если правильное и show==True)"""
        self.s.sendall(('check '+str(k)+'\n').encode())
        data=self.recv_until(b'\n')
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования Юникода. Попытайтесь переподключиться к серверу.')
            return None
        if show:
            print (data)
        if data.find('flag')!=-1:
            return True
        else:
            data=self.recv_until(b'>')
            try:
                data=data.decode()
            except UnicodeDecodeError:
                print ('Ошибка декодирования Юникода. Попытайтесь переподключиться к серверу.')
                return None
            if show:
                print (data)
            return False

    def getPointMultiple(self,x,y, show=True):
        """Получить k*<отправленная_точка> с сервера"""
        self.s.sendall(('mult '+'('+str(x)+','+str(y)+')'+'\n').encode())
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования Юникода. Попытайтесь переподключиться к серверу.')
            return None
        if show:
            print (data)
        if data.find('Infinity')!=-1:
            return (0,1,True)
        point=re.search(r'\(\d+,\d+\)',data).group(0)
        (x,y)=tuple(map(int,point[1:-1].split(',')))
        return (x,y,False)


    def __del__(self):
        self.s.close()

vs=VulnServerClient()
(Gx,Gy,p,a,b,group_order,Qx,Qy)=vs.getChallenge()


Welcome to Wrong Curve Attack task
I will be using a generator G=(39373638796100256138430072333893083119819913369338245519023152912113804320788,35664822287024859483374069777729332837629627782606926242774373200543075306102) on E(GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)) for y**2=x**3+4*x+221 which creates a group of order 113311125625150355228527808652129931021724931561015845843493194156576487486212
Q=kG
Q=(45888711352767259643115068244363596536394624880879892763098764014883435090799,26482520812742009842212417303003441700922501598629821600361345154970882016808) on E(GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)) for y**2=x**3+4*x+221
Commands:
help - show this banner
mult (<x>,<y>) - return this point multipled by secret k (e.g. "mult (-1,1)")
check <k> - check solution
Find k and send it to me
>


In [3]:
curve=EllipticCurve(p,a,b)
G=Point(curve,Gx,Gy)
Q=Point(curve,Qx,Qy)
#You can add points:
print (G+G)
#And you can multiply them
print (G*group_order)

(54986703416958884648633391095313408517752639158036668799371782683095459789958,29936481408830724865050321608435589234856945746004323468332935455983684427152) on E(GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)) for y**2=x**3+4*x+221
Точка на бесконечности на E(GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)) for y**2=x**3+4*x+221


In [4]:
vulnerable_curves=[
    #(A, B, order)
    (4, 142, 113311125625150355228527808652129931021494535848207968629801995728793690675887),
    (4, 254, 113311125625150355228527808652129931021927787708044781007210395786720384091850),
    (4, 257, 113311125625150355228527808652129931021320700166477893912982394278528617798310),
    (4, 277, 113311125625150355228527808652129931022405298874054957089156266046101116988928),
    (4, 286, 113311125625150355228527808652129931021623590516711700884680922146557735250848)
]

Get factors:

In [15]:
def get_prime_factors(n, limit=65536):
    factors = set()
    d = 2
    temp = n
    while d * d <= temp and d < limit:
        if temp % d == 0:
            factors.add(d)
            while temp % d == 0:
                temp //= d
        d += 1
    if temp > 1 and temp < limit:
        factors.add(temp)
    return list(factors)

Find point on curve:

In [6]:
import random

def find_point_on_curve(curve, total_order, factor):
    cofactor = total_order // factor
    while True:
        x = random.randint(0, curve.p - 1)

        rhs = (pow(x, 3, curve.p) + curve.a * x + curve.b) % curve.p

        if pow(rhs, (curve.p - 1) // 2, curve.p) != 1:
            continue

        y = pow(rhs, (curve.p + 1) // 4, curve.p)

        R = Point(curve, x, y)
        P = R * cofactor

        if not P.is_identity():
            return P

Solve DLP using bruteforce

In [7]:
def solve_dlp(curve, P, Q, factor):

    current = Point(curve=curve, x=0, y=1, is_identity=True)
    for i in range(factor):
        if current == Q:
            return i
        else:
            current = current + P

    return None


Full attack

In [17]:
import random
from sympy.ntheory.modular import crt


vs = VulnServerClient()
(Gx, Gy, p, a, b, group_order, Qx, Qy) = vs.getChallenge()

vulnerable_curves = [
    # (A, B, Order)
    (4, 142, 113311125625150355228527808652129931021494535848207968629801995728793690675887),
    (4, 254, 113311125625150355228527808652129931021927787708044781007210395786720384091850),
    (4, 257, 113311125625150355228527808652129931021320700166477893912982394278528617798310),
    (4, 277, 113311125625150355228527808652129931022405298874054957089156266046101116988928),
    (4, 286, 113311125625150355228527808652129931021623590516711700884680922146557735250848)
]

all_remainders = []
all_moduli = []

print(f"Server P: {p}")
print(f"Server A: {a}")

for A_vuln, B_vuln, N_vuln in vulnerable_curves:
    print(f"\nProcessing curve B={B_vuln}, Order={N_vuln}")

    curve_instance = EllipticCurve(p, A_vuln, B_vuln)

    small_factors = get_prime_factors(N_vuln)
    print(f"Small factors of order: {small_factors}")

    for r in small_factors:
        if r in all_moduli:
            continue

        print(f"Trying factor r={r}...", end=" ", flush=True)

        try:
            P_r = find_point_on_curve(curve_instance, N_vuln, r)

            res = vs.getPointMultiple(P_r.x, P_r.y, show=False)
            if res is None:
                print("Error getting response")
                continue

            rx, ry, is_inf = res

            if is_inf:
                k_mod_r = 0
                print(f"Infinity -> k mod {r} = 0")
            else:
                Q_r = Point(curve_instance, rx, ry)
                k_mod_r = solve_dlp(curve_instance, P_r, Q_r, r)
                if k_mod_r is not None:
                    print(f"Found k mod {r} = {k_mod_r}")
                else:
                    print("DLP failed")
                    continue

            if k_mod_r is not None:
                all_remainders.append(k_mod_r)
                all_moduli.append(r)

        except Exception as e:
            print(f"Exception: {e}")
            continue

if not all_moduli:
    print("No remainders collected.")
else:
    print(f"\nCollected {len(all_moduli)} remainders.")
    print(f"Moduli: {all_moduli}")

    k_recovered, lcm_val = crt(all_moduli, all_remainders)
    k_recovered %= lcm_val

    print(f"Recovered key: {k_recovered}")

    vs.checkSolution(k_recovered)

vs.s.close()

Welcome to Wrong Curve Attack task
I will be using a generator G=(39373638796100256138430072333893083119819913369338245519023152912113804320788,35664822287024859483374069777729332837629627782606926242774373200543075306102) on E(GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)) for y**2=x**3+4*x+221 which creates a group of order 113311125625150355228527808652129931021724931561015845843493194156576487486212
Q=kG
Q=(40775275221630526875311750070354531178908867287183695122861846499651345852115,20893638237315388956883346192571995163307493107412193939943366331737314151663) on E(GF(113311125625150355228527808652129931021748019940935843147703952255126187098699)) for y**2=x**3+4*x+221
Commands:
help - show this banner
mult (<x>,<y>) - return this point multipled by secret k (e.g. "mult (-1,1)")
check <k> - check solution
Find k and send it to me
>
Server P: 113311125625150355228527808652129931021748019940935843147703952255126187098699
Server A: 4

Processing c